In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
import warnings
warnings.filterwarnings('ignore')

class SFLA_FeatureSelection:
    def __init__(self, X, y, model_type='rf'):
        self.X = X
        self.y = y
        self.num_features = X.shape[1]

        # Adaptive parameters
        self.num_population = min(100, max(30, self.num_features // 10))
        self.num_memplexes = max(4, min(8, self.num_population // 10))
        self.max_iter = 150
        self.adaptive_step = True

        # Model for evaluation
        if model_type == 'rf':
            self.model = RandomForestClassifier(n_estimators=50, random_state=42)
        elif model_type == 'svm':
            self.model = SVC(kernel='linear', probability=True, random_state=42)

    def fitness_function(self, feature_indices):
        """Multi-objective fitness: Accuracy + Feature reduction"""
        if len(feature_indices) == 0:
            return float('inf')  # Penalize empty feature sets

        # Convert indices to binary representation
        if isinstance(feature_indices[0], bool):
            indices = np.where(feature_indices)[0]
        else:
            indices = feature_indices

        # Avoid duplicate indices
        indices = np.unique(indices)

        if len(indices) == 0:
            return float('inf')

        X_selected = self.X[:, indices]

        # Avoid too few samples for CV
        if len(indices) < 2:
            return float('inf')

        try:
            # Calculate accuracy using cross-validation
            scores = cross_val_score(self.model, X_selected, self.y,
                                   cv=min(5, len(np.unique(self.y))),
                                   scoring='accuracy', n_jobs=-1)
            accuracy = np.mean(scores)

            # Multi-objective: Maximize accuracy, minimize features
            alpha = 0.7  # Weight for accuracy
            beta = 0.3   # Weight for feature reduction

            # Normalize feature count penalty
            feature_penalty = len(indices) / self.num_features

            # Fitness to minimize (lower is better)
            fitness = 1 - (alpha * accuracy - beta * feature_penalty)

            # Add penalty for very small feature sets if accuracy is low
            if len(indices) < 5 and accuracy < 0.7:
                fitness += 0.5

            return fitness

        except Exception as e:
            return float('inf')

    def initialize_population(self):
        """Initialize population with diverse feature subsets"""
        population = []

        for _ in range(self.num_population):
            # Variable-length feature selection
            num_selected = np.random.randint(2, min(100, self.num_features // 2))
            indices = np.random.choice(self.num_features, num_selected, replace=False)
            population.append(indices)

        return population

    def adaptive_step_size(self, iteration, max_iter):
        """Adaptive step size for better convergence"""
        initial_step = 5
        final_step = 1
        return initial_step * (1 - iteration/max_iter) + final_step

    def local_search(self, frog, step_size):
        """Enhanced local search with mutation operations"""
        new_frog = frog.copy()

        if len(new_frog) == 0:
            new_frog = np.random.choice(self.num_features,
                                       np.random.randint(2, 10),
                                       replace=False)

        # Mutation operations
        mutation_type = np.random.choice(['add', 'remove', 'swap', 'perturb'])

        if mutation_type == 'add' and len(new_frog) < self.num_features - 1:
            # Add a feature
            available = np.setdiff1d(np.arange(self.num_features), new_frog)
            if len(available) > 0:
                new_feature = np.random.choice(available)
                new_frog = np.append(new_frog, new_feature)

        elif mutation_type == 'remove' and len(new_frog) > 2:
            # Remove a feature
            idx = np.random.randint(0, len(new_frog))
            new_frog = np.delete(new_frog, idx)

        elif mutation_type == 'swap' and len(new_frog) > 1:
            # Swap a feature
            idx = np.random.randint(0, len(new_frog))
            available = np.setdiff1d(np.arange(self.num_features), new_frog)
            if len(available) > 0:
                new_frog[idx] = np.random.choice(available)

        elif mutation_type == 'perturb':
            # Perturb indices
            perturbation = np.random.randint(-step_size, step_size + 1, size=len(new_frog))
            new_frog = (new_frog + perturbation) % self.num_features
            new_frog = np.unique(new_frog)  # Remove duplicates

        return np.unique(new_frog)  # Ensure no duplicates

    def optimize(self):
        """Main SFLA optimization loop"""
        # Initialize population
        population = self.initialize_population()
        best_global_fitness = float('inf')
        best_global_solution = None
        convergence_history = []

        for iteration in range(self.max_iter):
            # Adaptive step size
            if self.adaptive_step:
                current_step = self.adaptive_step_size(iteration, self.max_iter)
            else:
                current_step = 3

            # Evaluate fitness
            fitness = []
            for frog in population:
                fit = self.fitness_function(frog)
                fitness.append(fit)

            # Sort population
            sorted_indices = np.argsort(fitness)
            population = [population[i] for i in sorted_indices]
            fitness = [fitness[i] for i in sorted_indices]

            # Update global best
            if fitness[0] < best_global_fitness:
                best_global_fitness = fitness[0]
                best_global_solution = population[0].copy()

            # Shuffle and divide into memplexes
            np.random.shuffle(population)
            memplex_size = self.num_population // self.num_memplexes
            memplexes = [population[i:i + memplex_size]
                        for i in range(0, self.num_population, memplex_size)]

            # Local search in each memplex
            new_population = []
            for memplex in memplexes:
                memplex_fitness = [self.fitness_function(frog) for frog in memplex]
                best_idx = np.argmin(memplex_fitness)
                best_memplex_frog = memplex[best_idx]

                for frog in memplex:
                    # Generate new solution
                    new_frog = self.local_search(frog, current_step)
                    new_fitness = self.fitness_function(new_frog)

                    # Update if better
                    if new_fitness < self.fitness_function(frog):
                        frog = new_frog.copy()

                    # Learn from best in memplex
                    if np.random.rand() > 0.5:
                        # Combine features from best frog
                        combined = np.union1d(frog, best_memplex_frog)
                        if len(combined) > 0:
                            frog = combined.copy()

                    # Learn from global best
                    if np.random.rand() > 0.7 and best_global_solution is not None:
                        combined = np.union1d(frog, best_global_solution)
                        if len(combined) > 0:
                            frog = combined.copy()

                    new_population.append(frog.copy())

            population = new_population[:self.num_population]

            # Record convergence
            convergence_history.append(best_global_fitness)

            # Early stopping if convergence stalls
            if iteration > 20:
                if abs(convergence_history[-1] - np.mean(convergence_history[-10:])) < 0.001:
                    print(f"Early stopping at iteration {iteration}")
                    break

        return best_global_solution, best_global_fitness, convergence_history

# Usage example
if __name__ == "__main__":
    # Assuming X and y are your data
    # X shape: (n_samples, n_features)
    # y shape: (n_samples,)

    sfla_fs = SFLA_FeatureSelection(X, y, model_type='rf')
    best_features, best_fitness, history = sfla_fs.optimize()

    print(f"Selected {len(best_features)} features out of {X.shape[1]}")
    print(f"Best feature indices: {best_features}")
    print(f"Best fitness: {best_fitness}")